In [1]:
import pandas as pd
import numpy as np

# Load data
market = pd.read_csv("../backtesting/market_backtest_v4.csv")
market["Date"] = pd.to_datetime(market["Date"])
market = market.set_index("Date")

strategy_ret = market["strategy_ret_v2"]
benchmark_ret = market["nifty_ret"]

# ==================== BASIC RETURNS ====================
days = (market.index[-1] - market.index[0]).days

# CAGR
strategy_cagr = (market["strategy_equity"].iloc[-1] / market["strategy_equity"].iloc[0]) ** (365 / days) - 1
benchmark_cagr = (market["benchmark_equity"].iloc[-1] / market["benchmark_equity"].iloc[0]) ** (365 / days) - 1

print(f"Strategy CAGR:     {strategy_cagr:.2%}")
print(f"NIFTY CAGR:        {benchmark_cagr:.2%}")

# Volatility (annualized)
strategy_vol = strategy_ret.std() * np.sqrt(252)
benchmark_vol = benchmark_ret.std() * np.sqrt(252)
print(f"Strategy Volatility: {strategy_vol:.2%}")
print(f"NIFTY Volatility:    {benchmark_vol:.2%}")

# Sharpe Ratio (using ~6.5% rf)
rf_daily = 0.065 / 252
sharpe = (strategy_ret.mean() - rf_daily) / strategy_ret.std() * np.sqrt(252)
sharpe_nifty = (benchmark_ret.mean() - rf_daily) / benchmark_ret.std() * np.sqrt(252)

print(f"Strategy Sharpe:   {sharpe:.3f}")
print(f"NIFTY Sharpe:      {sharpe_nifty:.3f}")

# Sortino Ratio
downside = strategy_ret[strategy_ret < 0]
sortino = (strategy_ret.mean() - rf_daily) / downside.std() * np.sqrt(252) if len(downside) > 0 else 0
print(f"Strategy Sortino:  {sortino:.3f}")

# Max Drawdown
rolling_max = market["strategy_equity"].cummax()
drawdown = (market["strategy_equity"] - rolling_max) / rolling_max
max_dd = drawdown.min()
print(f"Strategy Max DD:   {max_dd:.2%}")

# Calmar Ratio
calmar = strategy_cagr / abs(max_dd) if max_dd != 0 else 0
print(f"Strategy Calmar:   {calmar:.3f}")

# ==================== SUMMARY TABLE ====================
metrics = pd.DataFrame({
    "Metric": ["CAGR", "Volatility", "Sharpe", "Sortino", "Max Drawdown", "Calmar"],
    "Strategy": [strategy_cagr, strategy_vol, sharpe, sortino, max_dd, calmar],
    "NIFTY": [benchmark_cagr, benchmark_vol, sharpe_nifty, np.nan, 
              (market["benchmark_equity"]/market["benchmark_equity"].cummax()-1).min(), 
              benchmark_cagr / abs((market["benchmark_equity"]/market["benchmark_equity"].cummax()-1).min())]
})

print("\n=== PERFORMANCE SUMMARY ===")
print(metrics.round(4))

# Save
market.to_csv("../backtesting/market_backtest_v5.csv")
metrics.to_csv("../backtesting/performance_metrics.csv", index=False)

print("\n✅ Metrics saved to backtesting/performance_metrics.csv")

Strategy CAGR:     0.13%
NIFTY CAGR:        10.70%
Strategy Volatility: 17.00%
NIFTY Volatility:    18.07%
Strategy Sharpe:   -0.290
NIFTY Sharpe:      0.305
Strategy Sortino:  -0.386
Strategy Max DD:   -38.37%
Strategy Calmar:   0.003

=== PERFORMANCE SUMMARY ===
         Metric  Strategy   NIFTY
0          CAGR    0.0013  0.1070
1    Volatility    0.1700  0.1807
2        Sharpe   -0.2899  0.3048
3       Sortino   -0.3864     NaN
4  Max Drawdown   -0.3837 -0.3839
5        Calmar    0.0034  0.2787

✅ Metrics saved to backtesting/performance_metrics.csv
